***

Preparing Workspace

***

In [ ]:


## Importing packages ---

import numpy as np
import pandas as pd
import os
from tqdm import tqdm
import re
from datetime import date
import requests
import ast
import xlwt
from xlwt.Workbook import *
from pandas import ExcelWriter
import xlsxwriter
import time
import functools as ft
# pd.options.display.float_format = '{:.0f}'.format


## Setting file paths ---

user = os.getlogin()
path_users = os.path.join('C:\\Users', user)

path_sp   = os.path.join(path_users, 'Sacramento Area Council of Governments\Regional Monitoring and Reporting - Documents')
path_agol = os.path.join(path_sp, 'Process Revamp', 'Task 8. Reproduce Progress Report indicators', 'Indicator Data', 'Census Data')
path_main = os.path.join(path_sp, 'Data')

if user == 'jfontes':
    path_git     = os.path.join(path_users, 'Documents', 'Projects', 'Regional-Monitoring', 'Indicator_Gen')
    path_config0 = os.path.join(path_git, 'config')
    path_code    = os.path.join(path_git, 'Data', 'Census')
    path_config  = os.path.join(path_code, 'config')


## User defined functions ---

exec(open(os.path.join(path_config0, 'Functions.py')).read())


## API key ---

# Obtain API Key from the following source 
# https://api.census.gov/data/key_signup.html
# Copy retrieved API key to .txt file for safe keeping
file_api = open(os.path.join(path_config, 'api_key.txt'))
api_key = file_api.read()
file_api.close()



***

Importing

***

In [ ]:


year_start = 2023
year_end   = 2023
years_to_import = range(year_start, year_end+1)



In [ ]:


start_time = time.time()

list_df_census = []

print('Importing place IDs for all states...')
print('')

for year in tqdm(years_to_import):


    # User inputs for user API key, desired variables
    # Specify which geography to import
    # Concatenate constructed URL
    # Call data using URL
    # Use requests package to call out to the API
    # convert parsed response text to pandas df
    # apply year tag
    
    root_ = f'https://api.census.gov/data/{year}/acs/acs5'
    g_ = '?get='
    variables_ = 'NAME'
    location_ = '&for=metropolitan%20statistical%20area/micropolitan%20statistical%20area:*'
    api_key_ = f"&key={api_key}"

    query = f"{root_}{g_}{variables_}{location_}{api_key_}"

    response = requests.get(query).text
    response = ast.literal_eval(response)
    
    df_census = pd.DataFrame(response[1:], columns = response[0])    
    df_census['Year'] = year

    list_df_census.append(df_census)


print('')
print('Concatenating all states together...')

df_msa = pd.concat(list_df_census)


print("")
print("Finished!!")
print(f"Process complete.  It took --- {round((time.time() - start_time)/60, 1)} minutes ---")



In [ ]:
df_msa

In [ ]:


## Old method
# # Import County FIPS workbook
# df_fips = pd.read_excel(os.path.join(path_config0, 'Area Codes.xlsx')
#                                 , sheet_name = 'CountyFIPS'
#                                 , dtype = {'State FIPS': object, 'County FIPS': object})
# display(df_fips.head())

# # Extract State Abbreviation from MSA label
# def extract_state(text):
#     return text.split(',')[1].strip()
    
# df_msa['State'] = df_msa['NAME'].apply(extract_state).str[:2]
# display(df_msa.head())

# # Merge State FIPS code onto MSA labels
# df_msa2 = df_msa.merge(df_fips[['State', 'State FIPS']].drop_duplicates(), on = 'State', how = 'left')
# df_msa2 = df_msa2.sort_values(['NAME']).drop_duplicates()
# df_msa2


## New method
df_config = pd.read_excel(os.path.join(path_config0, 'Area Codes.xlsx'), sheet_name='MSAcodes', dtype = {'MSA_ID': object, 'State FIPS': object})

df_msa = df_msa.rename(columns = {'NAME':'MSA', 'metropolitan statistical area/micropolitan statistical area':'MSA_ID'})
df_msa = df_msa.drop('Year', axis=1)
df_msa = df_msa.merge(df_config, on=['MSA_ID', 'MSA'], how='left')
df_msa.head()



***

Exporting

***

In [ ]:


# with pd.ExcelWriter(os.path.join(path_config0, 'Area Codes.xlsx'),mode='a',engine='openpyxl',if_sheet_exists='replace') as writer:
#             df_msa2.to_excel(writer, index = False, sheet_name = 'MSAcodes')

